In [6]:
import os
import re
import math
import pandas as pd
import numpy as np
import lightgbm as lgb
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, roc_auc_score
from sentence_transformers import SentenceTransformer

print("====================================================")
print("     PIPELINE 1: REPO_FILE (LightGBM + PCA)         ")
print("====================================================")

BASE_DIR = "../scripts/m_data/processed/repo_file" # Adjust path if needed
device = "mps" # or "mps" / "cpu"

# --- Linguistic Extractors ---
ROLEPLAY_OVERRIDE_PATTERNS = re.compile(r"(pretend you are|you are now|act as|dan mode|ignore (previous|all) instructions|disregard your rules|override)", re.IGNORECASE)
PERSISTENCE_POISONING_PATTERNS = re.compile(r"(write to|save this rule|remember|store this|update configuration|persist|memory/)", re.IGNORECASE)
EXFILTRATION_TRIGGER_PATTERNS = re.compile(r"(print the system|leak api|send contents|webhook|exfiltrate|unauth_disclosure)", re.IGNORECASE)
OBFUSCATION_PATTERNS = re.compile(r"(?:[A-Za-z0-9+/]{4}){2,}(?:[A-Za-z0-9+/]{2}==|[A-Za-z0-9+/]{3}=)?|\\u[0-9a-fA-F]{4}", re.IGNORECASE)

def calc_features(text):
    t = str(text) if not pd.isna(text) else ""
    t_len = max(len(t), 1)
    words = re.findall(r'\b\w+\b', t.lower())
    counts = Counter(t)
    freqs = [c / t_len for c in counts.values()]
    entropy = -sum(p * math.log2(p) for p in freqs) if freqs else 0.0
    ttr = len(set(words)) / len(words) if words else 0.0
    
    return {
        "text_length": len(t), "shannon_entropy": entropy, "payload_ttr": ttr,
        "caps_ratio": sum(1 for c in t if c.isupper()) / t_len,
        "punct_density": sum(1 for c in t if c in "!?.,;:#@*[]{}") / t_len,
        "obfuscation_flag": int(bool(OBFUSCATION_PATTERNS.search(t))),
        "roleplay_override_flag": int(bool(ROLEPLAY_OVERRIDE_PATTERNS.search(t))),
        "persistence_poisoning_flag": int(bool(PERSISTENCE_POISONING_PATTERNS.search(t))),
        "exfiltration_trigger_flag": int(bool(EXFILTRATION_TRIGGER_PATTERNS.search(t)))
    }

# 1. Load Data
train_df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))
print(f"[*] Loaded Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")

# 2. Extract NLP Metrics
print("[*] Extracting Structural Metrics...")
train_feats = pd.DataFrame(train_df["text"].apply(calc_features).tolist())
test_feats = pd.DataFrame(test_df["text"].apply(calc_features).tolist())

# 3. Embed & PCA
print("[*] Embedding Text (all-MiniLM-L6-v2)...")
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

train_embs = encoder.encode(train_df["text"].fillna("").tolist(), batch_size=128, show_progress_bar=True)
test_embs = encoder.encode(test_df["text"].fillna("").tolist(), batch_size=128, show_progress_bar=True)

print("[*] Fitting PCA (50 components)...")
pca = PCA(n_components=50, random_state=42)
train_pca = pd.DataFrame(pca.fit_transform(train_embs), columns=[f"pca_text_{i}" for i in range(50)])
test_pca = pd.DataFrame(pca.transform(test_embs), columns=[f"pca_text_{i}" for i in range(50)])

# 4. Assemble & Train
X_train = pd.concat([train_feats, train_pca], axis=1)
y_train = train_df["label"].values

X_test = pd.concat([test_feats, test_pca], axis=1)
y_test = test_df["label"].values

print(f"[*] Final Training Matrix Shape: {X_train.shape}")
clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)

# 5. Evaluate
y_pred = clf.predict(X_test)
print("\n" + "="*50 + "\n REPO_FILE LIGHTGBM REPORT\n" + "="*50)
print(classification_report(y_test, y_pred, target_names=["Benign", "Malicious"]))

     PIPELINE 1: REPO_FILE (LightGBM + PCA)         
[*] Loaded Train Shape: (4535, 7) | Test Shape: (1134, 7)
[*] Extracting Structural Metrics...
[*] Embedding Text (all-MiniLM-L6-v2)...


Batches: 100%|██████████| 9/9 [00:01<00:00,  7.55it/s]


[*] Fitting PCA (50 components)...
[*] Final Training Matrix Shape: (4535, 59)
[LightGBM] [Info] Number of positive: 2333, number of negative: 2202
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001109 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13861
[LightGBM] [Info] Number of data points in the train set: 4535, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000

 REPO_FILE LIGHTGBM REPORT
              precision    recall  f1-score   support

      Benign       0.89      0.90      0.90       551
   Malicious       0.91      0.89      0.90       583

    accuracy                           0.90      1134
   macro avg       0.90      0.90      0.90      1134
weighted avg       0.90      0.90      0.90      1134



In [7]:
import os
import joblib

SAVE_DIR = "./saved_repo_file_model"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(clf, os.path.join(SAVE_DIR, "lightgbm.pkl"))
joblib.dump(pca, os.path.join(SAVE_DIR, "pca.pkl"))
joblib.dump(list(X_train.columns), os.path.join(SAVE_DIR, "feature_columns.pkl"))

encoder.save(os.path.join(SAVE_DIR, "sentence_transformer"))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.68it/s]


In [3]:

print("\n====================================================")
print("  PIPELINE 2: INDIRECT_CONTEXT (LightGBM + PCA)     ")
print("====================================================")

BASE_DIR = "../scripts/m_data/processed/indirect_context"
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

train_df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))
print(f"[*] Loaded Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")

print("[*] Extracting Structural Metrics on primary text...")
train_feats = pd.DataFrame(train_df["text"].apply(calc_features).tolist())
test_feats = pd.DataFrame(test_df["text"].apply(calc_features).tolist())

print("[*] Embedding 'text' and 'paired_text'...")
train_text_embs = encoder.encode(train_df["text"].fillna("").tolist(), batch_size=256, show_progress_bar=True)
test_text_embs = encoder.encode(test_df["text"].fillna("").tolist(), batch_size=256, show_progress_bar=True)

train_paired_embs = encoder.encode(train_df["paired_text"].fillna("").tolist(), batch_size=256, show_progress_bar=True)
test_paired_embs = encoder.encode(test_df["paired_text"].fillna("").tolist(), batch_size=256, show_progress_bar=True)

print("[*] Fitting Dual PCAs (50 + 50 components)...")
pca_text = PCA(n_components=50, random_state=42)
train_pca_txt = pd.DataFrame(pca_text.fit_transform(train_text_embs), columns=[f"pca_txt_{i}" for i in range(50)])
test_pca_txt = pd.DataFrame(pca_text.transform(test_text_embs), columns=[f"pca_txt_{i}" for i in range(50)])

pca_paired = PCA(n_components=50, random_state=42)
train_pca_pair = pd.DataFrame(pca_paired.fit_transform(train_paired_embs), columns=[f"pca_pair_{i}" for i in range(50)])
test_pca_pair = pd.DataFrame(pca_paired.transform(test_paired_embs), columns=[f"pca_pair_{i}" for i in range(50)])

# Assemble
X_train = pd.concat([train_feats, train_pca_txt, train_pca_pair], axis=1)
y_train = train_df["label"].values

X_test = pd.concat([test_feats, test_pca_txt, test_pca_pair], axis=1)
y_test = test_df["label"].values

print(f"[*] Final Training Matrix Shape: {X_train.shape}")
clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("\n" + "="*50 + "\n INDIRECT_CONTEXT LIGHTGBM REPORT\n" + "="*50)
print(classification_report(y_test, y_pred, target_names=["Benign", "Malicious"]))


  PIPELINE 2: INDIRECT_CONTEXT (LightGBM + PCA)     


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8388.28it/s]


[*] Loaded Train Shape: (56000, 7) | Test Shape: (14000, 7)
[*] Extracting Structural Metrics on primary text...
[*] Embedding 'text' and 'paired_text'...


Batches: 100%|██████████| 55/55 [00:30<00:00,  1.81it/s]


[*] Fitting Dual PCAs (50 + 50 components)...
[*] Final Training Matrix Shape: (56000, 109)
[LightGBM] [Info] Number of positive: 28000, number of negative: 28000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012310 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 26665
[LightGBM] [Info] Number of data points in the train set: 56000, number of used features: 108
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

 INDIRECT_CONTEXT LIGHTGBM REPORT
              precision    recall  f1-score   support

      Benign       0.90      0.85      0.87      7000
   Malicious       0.86      0.90      0.88      7000

    accuracy                           0.88     14000
   macro avg       0.88      0.88      0.88     14000
weighted avg       0.88      0.88      0.88     14000



In [4]:
import os
import joblib

SAVE_DIR = "./saved_indirect_context_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save LightGBM model
joblib.dump(clf, os.path.join(SAVE_DIR, "lightgbm.pkl"))

# Save PCA models
joblib.dump(pca_text, os.path.join(SAVE_DIR, "pca_text.pkl"))
joblib.dump(pca_paired, os.path.join(SAVE_DIR, "pca_paired.pkl"))

# Save feature column order
joblib.dump(list(X_train.columns),
            os.path.join(SAVE_DIR, "feature_columns.pkl"))

# Save the SentenceTransformer locally
encoder.save(os.path.join(SAVE_DIR, "sentence_transformer"))

print("Model saved successfully!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.19it/s]

Model saved successfully!


In [ ]:
# import joblib
# from sentence_transformers import SentenceTransformer

# MODEL_DIR = "./saved_indirect_context_model"

# # Load models
# clf = joblib.load(os.path.join(MODEL_DIR, "lightgbm.pkl"))
# pca_text = joblib.load(os.path.join(MODEL_DIR, "pca_text.pkl"))
# pca_paired = joblib.load(os.path.join(MODEL_DIR, "pca_paired.pkl"))
# feature_columns = joblib.load(os.path.join(MODEL_DIR, "feature_columns.pkl"))

# encoder = SentenceTransformer(
#     os.path.join(MODEL_DIR, "sentence_transformer")
# )

In [1]:
import psutil

print(f"Available RAM: {psutil.virtual_memory().available / 1024**3:.2f} GB")
print(f"Total RAM: {psutil.virtual_memory().total / 1024**3:.2f} GB")

Available RAM: 6.20 GB
Total RAM: 16.00 GB


In [5]:
import sys
import torch
import sentence_transformers
import transformers

print(sys.version)
print(torch.__version__)
print(transformers.__version__)
print(sentence_transformers.__version__)

/Users/inviforce/Downloads/VS_CODE_C++_PYTHON_JUPYTER_DEV_/LLM Guardian/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


3.14.4 (main, Apr  7 2026, 13:13:20) [Clang 21.0.0 (clang-2100.0.123.102)]
2.13.0
5.13.1
5.6.0


In [ ]:
# =====================================================
# SAVE EVERYTHING NEEDED FOR INFERENCE
# =====================================================

print("\nSaving model artifacts...")

SAVE_DIR = "saved_agent_session_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# LightGBM model
joblib.dump(clf, os.path.join(SAVE_DIR, "lightgbm_model.pkl"))

# PCA
joblib.dump(pca, os.path.join(SAVE_DIR, "pca.pkl"))

# Feature list
joblib.dump(numeric_cols, os.path.join(SAVE_DIR, "numeric_features.pkl"))

# Encoder name
joblib.dump("all-MiniLM-L6-v2", os.path.join(SAVE_DIR, "encoder_name.pkl"))

print(f"\nAll artifacts saved to: {SAVE_DIR}")

print("""
Saved Files:
-------------
lightgbm_model.pkl
pca.pkl
numeric_features.pkl
encoder_name.pkl
""")